In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "../01_setup/03_config"

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://child-company-data/{data_source}'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)


# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

Base Path:  s3://child-company-data/orders
Landing Path:  s3://child-company-data/orders/landing/
Processed Path:  s3://child-company-data/orders/processed/


# Bronze Processing

In [0]:
files = dbutils.fs.ls(landing_path)

if len(files) == 0:
    print("No new files to process. Exiting job.")
    dbutils.notebook.exit("NO_DATA")
else:
    df = spark.read.format("csv") \
        .option("header", True) \
        .load(f"{landing_path}/*.csv") \
        .withColumn("read_timestamp", F.current_timestamp()) \
        .select("*", "_metadata.file_name", "_metadata.file_size")

    print("Total Rows:", df.count())
    display(df.limit(20))

    df.write \
        .format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .mode("append") \
        .saveAsTable(bronze_table)

    # move files only after successful write
    for f in files:
        dbutils.fs.mv(
            f.path,
            f"{processed_path}/{f.name}",
            True
        )


No new files to process. Exiting job.


# Silver Processing

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
display(df_orders.limit(5))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FJUL33320501,2025/07/01,789320,25891203,150.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL33320501,2025/07/01,789320,25891301,46.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL33320501,01-07-2025,789320,25891403,null,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL33320501,"Tuesday, July 01, 2025",789320,25891201,354.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL33320501,"Tuesday, July 01, 2025",789320,25891501,249.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744


In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
display(df_orders.limit(10))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FJUL33320501,2025-07-01,789320,25891203,150.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891301,46.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891201,354.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891501,249.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891302,40.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891502,133.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891503,145.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891203,429.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891201,461.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744
FJUL32101601,2025-07-01,789101,25891503,183.0,2025-12-31T05:35:58.668Z,orders_2025_07_01.csv,20744


In [0]:
df_products = spark.table("fmcg.silver.products")
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

display(df_joined.limit(5))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size,product_code
FJUL36503603,2025-07-05,789503,25891102,426.0,2025-12-31T05:35:58.668Z,orders_2025_07_05.csv,20520,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
FJUL319521103,2025-07-17,789521,25891103,331.0,2025-12-31T05:35:58.668Z,orders_2025_07_17.csv,19875,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268
FJUL321622501,2025-07-18,789622,25891303,50.0,2025-12-31T05:35:58.668Z,orders_2025_07_18.csv,18495,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f
FJUL322603603,2025-07-20,789603,25891601,104.0,2025-12-31T05:35:58.668Z,orders_2025_07_20.csv,19304,716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742
FJUL325201601,2025-07-22,789201,25891601,200.0,2025-12-31T05:35:58.668Z,orders_2025_07_22.csv,20690,716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742


In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# Gold Processing

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

display(df_gold.limit(2))

order_id,date,customer_code,product_code,product_id,sold_quantity
FSEP59903603,2025-09-06,789903,451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,25891603,91.0
FSEP59903601,2025-09-08,789903,716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,25891601,86.0


In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# Merging Data Source with Parent

In [0]:
#Note: We want data for monthly level but child data is on daily level
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(10)

+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-08-08|102628255d24304d6...|       789101|        493.0|
|2025-08-08|889c67757ece9c973...|       789101|        374.0|
|2025-08-08|d9ebd1ca64d23951a...|       789101|         44.0|
|2025-08-07|e91ba9d665f90254d...|       789101|        311.0|
|2025-08-07|2e387cef1424d6e7b...|       789101|        442.0|
|2025-08-07|fe5a8036be4b9a787...|       789101|        239.0|
|2025-08-09|c68834ceaff15846b...|       789101|         23.0|
|2025-08-09|ee1f7df9cf660ef02...|       789101|        123.0|
|2025-08-08|451f7167b28a25bde...|       999999|        197.0|
|2025-08-07|e91ba9d665f90254d...|       789102|        333.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows


In [0]:
df_child.count()

40811

In [0]:
df_monthly = (
    df_child
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-08-01|102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268|789101       |6956.0       |
|2025-08-01|889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2|789101       |2673.0       |
|2025-08-01|d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5|789101       |828.0        |
|2025-08-01|e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843|789101       |3732.0       |
|2025-08-01|2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896|789101       |3522.0       |
+----------+----------------------------------------------------------------+-------------+-------------+
only showing top 5 rows


In [0]:
df_monthly.count()

3060

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]